# **Spoken Digit Recognition**

In [1]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping


In [2]:
from utils.preprocess import preprocess
from utils.features import extract_mfcc

## Load dataset

In [3]:
DATASET_PATH = "dataset/"

labels = []
features = []

for digit in os.listdir(DATASET_PATH):

    digit_path = os.path.join(DATASET_PATH, digit)

    for file in os.listdir(digit_path):

        file_path = os.path.join(digit_path, file)

        # load audio
        audio, sr = librosa.load(
            file_path,
            sr=16000
        )

        labels.append(digit)
        features.append(audio)

print("Total Samples:", len(features))

C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total Samples: 17000


## Preprocessing & Feature Extraction

In [4]:
X = []
y = []

for audio, label in zip(features, labels):
    audio = preprocess(audio)
    mfcc = extract_mfcc(audio)
    
    X.append(mfcc)
    y.append(label)

X = np.array(X)
X = X[..., np.newaxis]  # CNN channel

le = LabelEncoder()
y = le.fit_transform(y)
y = to_categorical(y)

## Train/Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Build Model

In [6]:
model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=X_train.shape[1:]))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(10, activation='softmax'))

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

c:\Program Files\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

## Training

In [12]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/50
425/425 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9336 - loss: 0.2029 - val_accuracy: 0.9129 - val_loss: 0.2744
Epoch 2/50
425/425 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9449 - loss: 0.1700 - val_accuracy: 0.9168 - val_loss: 0.2703
Epoch 3/50
425/425 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9454 - loss: 0.1593 - val_accuracy: 0.9168 - val_loss: 0.3204
Epoch 4/50
425/425 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9539 - loss: 0.1333 - val_accuracy: 0.9212 - val_loss: 0.3320
Epoch 5/50
425/425 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9572 - loss: 0.1257 - val_accuracy: 0.9209 - val_loss: 0.2985
Epoch 6/50
425/425 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9615 - loss: 0.1123 - val_accuracy: 0.9297 - val_loss: 0.2486
Epoch 7/50
425/425 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9648 - loss: 0.1081 - val_accuracy: 0.9194 - val_loss: 0.3224
Epoch 8/50
425/425 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9651 - loss: 0.1068 - val_accura

In [13]:
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("\nTest Accuracy:", accuracy)

107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9297 - loss: 0.2486

Test Accuracy: 0.9297058582305908


In [15]:
model.save("models/spoken_digit_model.keras")